# 🟡 KaizenStat — Intermediate Demo (15 min)

**Level:** Intermediate | **Time:** ~15 minutes | **Dataset:** Titanic

This notebook covers the **full 8-step pipeline** plus:
- Root-cause model debugging
- Production Trust Score
- Pipeline Confidence Score
- Understanding improvement suggestions
- Before vs After with `auto_improve()`

---
> **Prerequisites:** Basic pandas/sklearn knowledge
>
> **What you'll learn:**
> - How to diagnose *why* your model fails (data vs model problem)
> - How to get a production readiness score
> - How to interpret improvement suggestions with priority levels

In [ ]:
!pip install kaizenstat -q
print("✅ KaizenStat installed")

## Setup — Load and Inspect

We drop `PassengerId`, `Name`, `Ticket`, and `Cabin` — high-cardinality ID/text columns
with no predictive value.

In [ ]:
import pandas as pd
import numpy as np
from kaizenstat import DataDoctor

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

# Drop ID / free-text columns
df = df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])

print("=" * 50)
print("TITANIC DATASET")
print("=" * 50)
print(f"Shape: {df.shape}")
print(f"\nTarget distribution (Survived):")
print(df['Survived'].value_counts())
print(f"\nMissing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])
df.head()

## Full 8-Step Pipeline

In [ ]:
# Step 1 — fit
doctor = DataDoctor()
doctor.fit(df, target="Survived")

In [ ]:
# Step 2 — health
health = doctor.health()
print(f"\nHealth Score: {health.score} / 100")
print("  Age: 19.9% missing → moderate penalty")
print("  Embarked: 0.2% missing → minor penalty")

In [ ]:
# Step 3 — validate
# Checks leakage, drift, multicollinearity
validation = doctor.validate()
print(f"\nIssues found: {len(validation.issues)}")

In [ ]:
# Step 4a — preview only (no changes)
print("PREVIEW — no changes applied yet")
doctor.fix(safe=True, preview_only=True)

In [ ]:
# Step 4b — apply fixes
fixed_df = doctor.fix(safe=True)
print(f"\nAfter fix: {fixed_df.isnull().sum().sum()} missing values")

In [ ]:
# Step 5 — train
train_result = doctor.train(cv=5)

print(f"\n{'='*40}")
print(f"Best model:  {train_result.model_name}")
print(f"Test score:  {train_result.test_score:.4f}")
print(f"Train score: {train_result.train_score:.4f}")
print(f"Gap:         {train_result.train_score - train_result.test_score:.4f}")

### Step 6 — debug_model()

This is where KaizenStat is **uniquely valuable**.

`debug_model()` answers: *Is the low score caused by bad data, or the wrong model choice?*

It also shows feature importances and which subgroups the model gets wrong most often.

In [ ]:
debug_result = doctor.debug_model()

print(f"\n{'='*40}")
print("DEBUG RESULTS")
print(f"{'='*40}")
print(f"Train score: {debug_result.train_score:.4f}")
print(f"Test score:  {debug_result.test_score:.4f}")
print(f"Gap:         {debug_result.gap:.4f}")

gap = debug_result.gap
if gap > 0.15:
    print("\n🔴 VERDICT: Overfitting — try regularisation or more data")
elif gap < -0.05:
    print("\n🔴 VERDICT: Underfitting — try a more complex model")
else:
    print("\n🟢 VERDICT: Healthy generalisation")

In [ ]:
# Feature importances
print("Counterfactual feature impact (score drop when feature removed):")
impact = doctor.feature_impact(top_n=8)
for feat, drop in sorted(impact.items(), key=lambda x: -x[1]):
    bar = '█' * max(1, int(drop * 150))
    print(f"  {feat:20s}  {drop:.4f}  {bar}")

In [ ]:
# Step 7 — improve
improvement_report = doctor.improve()

print("\n--- Recommended next actions ---")
actions = doctor.recommend_actions()
for i, action in enumerate(actions[:5], 1):
    print(f"  {i}. {action}")

### Trust Score — Production Readiness

```
trust_score = 0.40 × accuracy
            + 0.25 × robustness
            + 0.20 × calibration
            + 0.15 × certainty
```

**Threshold: ≥ 75 = production-ready.**

In [ ]:
trust = doctor.trust_score()

print(f"\n{'='*40}")
print(f"Trust Score: {trust.score:.0f} / 100")
if trust.score >= 75:
    print("✅ PRODUCTION READY")
elif trust.score >= 60:
    print("🟡 Needs improvement before production")
else:
    print("🔴 NOT production ready")

In [ ]:
confidence = doctor.pipeline_confidence()
print(f"\nPipeline Confidence: {confidence} / 100")

In [ ]:
# Step 8 — report
report_path = doctor.report(output_path="intermediate_report.html")
print(f"📄 Report: {report_path}")

from IPython.display import IFrame, display
display(IFrame(src='intermediate_report.html', width='100%', height='600px'))

## Before vs After: auto_improve()

3-step automated loop: baseline train → apply safe fixes → retrain → compare.
Shows exact score delta from fixes.

In [ ]:
doctor2 = DataDoctor()
doctor2.fit(df, target="Survived")

comparison = doctor2.auto_improve(tune=False)

print(f"\n📈 Score delta: {comparison.score_delta:+.4f}")
if comparison.score_delta > 0:
    print(f"✅ Improvement from safe fixes: +{comparison.score_delta*100:.1f}%")
else:
    print("ℹ️  No gain — data already clean enough for safe fixes alone")

## 🎯 Exercises

**Exercise 1:** Check dataset difficulty
```python
difficulty = doctor.dataset_difficulty()
print(f"Difficulty: {difficulty:.3f}")  # 0=easy, 1=impossible
```

**Exercise 2:** Add a custom validation check
```python
def check_sex_column(df, target):
    if 'Sex' not in df.columns:
        return ["Missing 'Sex' — strong Titanic predictor"]
    return []

doctor.add_check(check_sex_column, name="sex_check")
doctor.validate()
```

**Exercise 3:** Enable tuning
```python
doctor3 = DataDoctor()
doctor3.fit(df, target="Survived")
doctor3.fix(safe=True)
tuned = doctor3.train(tune=True, n_iter=20)
print(f"Tuned: {tuned.test_score:.4f}  Params: {tuned.best_params}")
```

---
## What's Next?

| Notebook | Level | |
|----------|-------|-|
| [Basic (5 min)](demo_basic.ipynb) | 🟢 | fit → health → train |
| **You are here** | 🟡 | Full pipeline + debug + trust |
| [Advanced (30 min)](demo_advanced.ipynb) | 🔴 | Tuning + feature engineering + codegen |

---
*KaizenStat v0.5.1 · [GitHub](https://github.com/kaizenstat-python/KaizenStat) · MIT License*